# 🔐 AD Attack Path Analyzer — Automated Active Directory Attack Path Analysis

**Capstone project — Cybersecurity**

This notebook takes a **SharpHound** export (`.zip`) as input (the collector used by
[BloodHound](https://github.com/SpecterOps/BloodHound)), analyzes it **offline**, models Active
Directory as a graph, detects the shortest attack paths toward high-value targets, and then calls
an **LLM** (Groq, Anthropic Claude, or OpenAI, your choice) to explain each path in plain language
and propose Blue Team remediations.

### Notebook pipeline

| # | Step | Description |
|---|------|-------------|
| 1 | **Configuration** | Imports, API keys, global parameters |
| 2 | **Extraction & Parsing** | Reading the `.zip` in memory, extracting objects and critical relationships |
| 3 | **Graph modeling** | Building and cleaning a directed graph (`NetworkX`) |
| 4 | **Natural-language translation** | Turning a graph path into a readable paragraph |
| 5 | **LLM analysis (Red/Blue Team)** | Explaining the attack + remediations, via a structured prompt |

> ⚠️ **Legal & ethical scope.** This tool must only be used against Active Directory environments
> you are **explicitly authorized to audit** (a personal lab, a contracted pentest engagement, a
> CTF). SharpHound itself is an intrusive collection tool, so any use outside an authorized scope
> is illegal in most jurisdictions. This notebook only analyzes data that has **already been
> collected** — it performs no offensive action of its own.
>
> 📦 **Production version.** This version expects a **real** SharpHound export at
> `CONFIG["SHARPHOUND_ZIP_PATH"]` — there is no built-in demo/sample data. If the file is
> missing, the notebook raises a clear `FileNotFoundError` instead of silently falling back to
> anything else.


## 1. Configuration and imports

We only import standard building blocks plus `networkx` (graphs) and `pandas` (result tables).
The `ijson` library is **optional**: if present, it lets us parse SharpHound's JSON files **in
streaming mode** (without ever loading the whole file into memory), which matters for large AD
environments (tens of thousands of objects, JSON files that can reach several hundred MB). If it
is absent, the notebook automatically falls back to plain `json.load`.

```bash
pip install networkx pandas ijson openai anthropic  # openai covers both OpenAI and Groq (OpenAI-compatible endpoint)
```


In [ ]:
# On Google Colab, the following dependencies are not all preinstalled.
# Uncomment this line if needed (the openai package is used both for OpenAI directly and for
# Groq, which exposes an OpenAI-compatible endpoint — see section 5):
# !pip install -q networkx pandas ijson openai anthropic

import zipfile
import io
import os
import re
import json
import time
import warnings
from collections import Counter

import networkx as nx
import pandas as pd
from IPython.display import display

# --- Optional dependency: streaming JSON parsing for large volumes ---
try:
    import ijson
    IJSON_AVAILABLE = True
except ImportError:
    IJSON_AVAILABLE = False
    warnings.warn(
        "The 'ijson' library is not installed: parsing will fall back to json.load "
        "(less suited to very large files). Install it with: pip install ijson"
    )

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

print(f"NetworkX {nx.__version__} | Pandas {pd.__version__} | ijson streaming: {IJSON_AVAILABLE}")

In [ ]:
# ---------------------------------------------------------------------------------
# Global notebook settings — adjust to your context
# ---------------------------------------------------------------------------------

CONFIG = {
    # Path to your real SharpHound archive. This is REQUIRED in this production version:
    # there is no fallback demo dataset. If the file is missing, the notebook raises a
    # FileNotFoundError (see section 2) instead of generating fake data.
    "SHARPHOUND_ZIP_PATH": "sharphound_export.zip",

    # Hint (case-insensitive substring of the name) of the account considered already
    # compromised ("owned"), from which attack paths are searched.
    # SharpHound does NOT store this information (it's a BloodHound-UI-only concept), so it
    # must be provided manually here. Replace this placeholder with your actual scenario.
    "COMPROMISED_PRINCIPAL_HINT": "your_owned_account",

    # LLM provider: "auto" automatically detects the 1st available API key in the environment
    # (Anthropic, then OpenAI, then Groq), otherwise falls back to an offline "mock" mode.
    # Here we force Groq (free tier, OpenAI-compatible SDK, very low latency).
    "LLM_PROVIDER": "groq",          # "auto" | "anthropic" | "openai" | "groq" | "mock"
    "ANTHROPIC_MODEL": "claude-sonnet-5-5",
    "OPENAI_MODEL": "gpt-4o-mini",
    "GROQ_MODEL": "llama-3.3-70b-versatile",   # full catalog: https://console.groq.com/docs/models

    # Approximate token budget reserved per LLM call, used to group several short paths into
    # a single call without exceeding the model's context window.
    "MAX_TOKENS_PER_LLM_CALL": 3000,

    # Maximum length (number of hops) of an attack path to consider relevant.
    "MAX_PATH_LENGTH": 8,
}

# --- Google Colab integration (optional): securely retrieving the Groq API key ---
# On Colab: 🔑 "Secrets" icon in the left sidebar -> add a secret named GROQ_API_KEY
# (free key from https://console.groq.com/keys), then grant the notebook access to it.
# This block only runs when actually on Colab (otherwise it's silently skipped, so the
# notebook stays portable on local / JupyterLab setups).
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    pass  # Not on Colab: set GROQ_API_KEY another way (a regular environment variable)
except Exception as e:
    warnings.warn(f"Could not retrieve GROQ_API_KEY from Colab secrets: {e}")

# API keys are read from environment variables — NEVER hard-code them here, and never paste
# them into a cell, a shared notebook, or a Git repository (even a private one).
#
# To set yours BEFORE launching Jupyter (outside Colab), in a terminal:
#   export GROQ_API_KEY="your_key"           # Linux / macOS
#   setx GROQ_API_KEY "your_key"             # Windows (open a new prompt afterwards)
# ⚠️ If a key has ever been pasted in plain text anywhere (chat, screenshot, shared code),
# treat it as compromised and regenerate it in the provider's console.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

print("Anthropic key detected:", bool(ANTHROPIC_API_KEY))
print("OpenAI key detected   :", bool(OPENAI_API_KEY))
print("Groq key detected     :", bool(GROQ_API_KEY))

## 2. Extraction and parsing

SharpHound exports one JSON file per object type (`*_users.json`, `*_groups.json`,
`*_computers.json`, `*_domains.json`, `*_gpos.json`, `*_ous.json`, ...), each prefixed with a
timestamp that varies with every collection run. We only care about **users**, **groups**, and
**computers**, which carry the relationships exploitable for ACL-abuse and delegation attacks.

**Technical constraints handled:**
- 📦 **Zero disk writes**: the archive is read entirely in memory (`io.BytesIO`) and every member
  of the zip is opened as a stream (`zf.read(...)`), never physically extracted.
- 🐘 **Large volumes**: SharpHound's JSON files can exceed several hundred MB on large AD
  environments (tens of thousands of objects). Parsing uses `ijson` in streaming mode (object by
  object) when available, rather than `json.load`, which materializes the whole JSON tree in memory.
- 🧯 **Resilience**: a missing file in the zip, a corrupted zip, or invalid JSON in a single file
  (or a single object) must never crash the whole pipeline — only the affected part is skipped,
  with an explicit warning.
- 🚫 **No fallback data**: this production version strictly requires a real SharpHound export at
  `CONFIG["SHARPHOUND_ZIP_PATH"]`. If it's missing, a clear `FileNotFoundError` is raised.

**Relationships extracted** (the ones that actually matter for an attack path):

| Relationship | JSON source | Offensive meaning |
|---|---|---|
| `GenericAll` / `WriteDacl` / `ForceChangePassword` | `Aces[]` of the target object | ACL abuse: full control, permission changes, password reset |
| `AdminTo` | `LocalAdmins.Results[]` of `computers` | Local administrator rights on a machine |
| `HasSession` | `Sessions.Results[]` of `computers` | Active user session on a machine (potential LSASS dump) |
| `AllowedToDelegate` | `AllowedToDelegate[]` of `users`/`computers` | Kerberos constrained delegation, abusable to impersonate an identity |
| `MemberOf` | `Members[]` of `groups` | Group membership (privilege inheritance chain) |

> 💡 `GenericAll`, `WriteDacl`, `ForceChangePassword` are real ACE rights (`RightName` field),
> filtered via the `RELEVANT_ACE_RIGHTS` list below — **freely extensible**
> (e.g. `GenericWrite`, `Owns`, `WriteOwner`, `AddMember`, `AllowedToAct` for RBCD).
> `AdminTo`, `HasSession`, and `AllowedToDelegate`, on the other hand, are **not** ACEs: they come
> from dedicated JSON structures and are therefore handled by always-active dedicated code.


In [ ]:
# ---------------------------------------------------------------------------------
# Parsing constants
# ---------------------------------------------------------------------------------

# Patterns to locate the 3 files we care about, whatever their timestamp prefix
SHARPHOUND_FILE_PATTERNS = {
    "users": re.compile(r"users\.json$", re.IGNORECASE),
    "groups": re.compile(r"groups\.json$", re.IGNORECASE),
    "computers": re.compile(r"computers\.json$", re.IGNORECASE),
}

# ACE rights (the "RightName" field) considered relevant for ACL abuse.
# -> Freely extensible, for example:
#    RELEVANT_ACE_RIGHTS |= {"GenericWrite", "Owns", "WriteOwner", "AddMember", "AllowedToAct"}
RELEVANT_ACE_RIGHTS = {"GenericAll", "WriteDacl", "ForceChangePassword"}

OBJECT_TYPE_MAP = {"users": "User", "groups": "Group", "computers": "Computer"}

print("Monitored ACE rights:", RELEVANT_ACE_RIGHTS)

In [ ]:
def _iter_json_array(file_obj, member_name):
    '''
    Iterates over the objects of a SharpHound JSON file WITHOUT loading it entirely into memory
    (streaming via `ijson` when available). Handles two formats:
      - {"data": [...], "meta": {...}}   (recent SharpHound / BloodHound CE, >= v4)
      - [...]                             (legacy formats, flat list)

    Raises ValueError for a structurally invalid JSON file, but NOT for individual malformed
    objects (handled further up, in extract_objects_and_edges, so only that one object is lost
    rather than the whole file).
    '''
    if IJSON_AVAILABLE:
        for prefix in ("data.item", "item"):
            file_obj.seek(0)
            try:
                count = 0
                for obj in ijson.items(file_obj, prefix):
                    count += 1
                    yield obj
                if count > 0:
                    return
            except ijson.JSONError:
                continue
        warnings.warn(
            f"[{member_name}] Streaming with ijson found nothing (unexpected structure or "
            f"small file) -> falling back to json.load."
        )

    file_obj.seek(0)
    try:
        raw = json.load(file_obj)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {member_name}: {e}") from e

    if isinstance(raw, dict) and "data" in raw:
        yield from raw["data"]
    elif isinstance(raw, list):
        yield from raw
    else:
        raise ValueError(f"Unexpected structure in {member_name}: neither a 'data' key nor a flat list.")


def load_sharphound_zip(zip_bytes: bytes, verbose: bool = True):
    '''
    Opens a SharpHound archive IN MEMORY (no disk extraction) and locates the
    users/groups/computers.json files. Other files (gpos, ous, domains, containers...) are
    ignored: they don't carry the relationships this tool targets.

    Returns (raw_streams, member_names):
      - raw_streams  : {category: io.BytesIO} ready to parse
      - member_names : {category: file name inside the archive} (for error messages)

    Raises:
      - ValueError          if the archive is not a valid zip
      - FileNotFoundError   if no users/groups/computers.json file is found
    '''
    try:
        zf = zipfile.ZipFile(io.BytesIO(zip_bytes))
    except zipfile.BadZipFile as e:
        raise ValueError(f"Invalid or corrupted ZIP file: {e}") from e

    member_names = {}
    for category, pattern in SHARPHOUND_FILE_PATTERNS.items():
        matches = [n for n in zf.namelist() if pattern.search(n)]
        if not matches:
            warnings.warn(f"No '*{category}.json' file found in the archive — skipped.")
            continue
        if len(matches) > 1 and verbose:
            print(f"[info] Multiple files match '{category}': {matches} -> using {matches[0]}")
        member_names[category] = matches[0]

    if not member_names:
        raise FileNotFoundError(
            "No users/groups/computers.json file recognized in this zip. "
            "Check that this is indeed a valid SharpHound archive."
        )

    raw_streams = {}
    for category, member_name in member_names.items():
        try:
            data = zf.read(member_name)  # read into memory, never written to disk
        except (zipfile.BadZipFile, KeyError, RuntimeError) as e:
            warnings.warn(f"Could not read '{member_name}' from the archive: {e} — skipped.")
            continue
        raw_streams[category] = io.BytesIO(data)

    return raw_streams, member_names

In [ ]:
def _safe_get(d, *keys, default=None):
    '''Tolerant nested access: _safe_get(obj, "Sessions", "Results").'''
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k, default)
    return cur


def extract_objects_and_edges(raw_streams, member_names, relevant_rights=None, verbose=True):
    '''
    Walks the users/groups/computers streams and produces:
      - nodes : {sid: {"name", "type", "highvalue", "enabled", "admincount", "domain"}}
      - edges : [(source_sid, target_sid, relation, meta_dict), ...]
      - stats : useful counters (objects processed, errors skipped, relationship breakdown)

    Each object is processed in its own try/except: an error on ONE object (e.g. a malformed
    ACE) never interrupts processing of the next objects or of the other files.
    '''
    if relevant_rights is None:
        relevant_rights = RELEVANT_ACE_RIGHTS

    nodes, edges = {}, []
    stats = {"objects_parsed": 0, "parse_errors": 0, "edges_by_type": Counter()}

    def add_edge(src, dst, relation, **meta):
        if not src or not dst:
            return
        edges.append((src, dst, relation, meta))
        stats["edges_by_type"][relation] += 1

    for category, stream in raw_streams.items():
        member_name = member_names.get(category, category)
        default_type = OBJECT_TYPE_MAP.get(category, "Base")

        try:
            object_iter = _iter_json_array(stream, member_name)
        except ValueError as e:
            warnings.warn(f"Could not parse {member_name}: {e} — file skipped.")
            continue

        while True:
            try:
                obj = next(object_iter)
            except StopIteration:
                break
            except Exception as e:  # ijson stream interrupted mid-way -> stop this file
                warnings.warn(f"Stream interrupted for {member_name}: {e}")
                break

            try:
                oid = obj.get("ObjectIdentifier")
                if not oid:
                    continue
                props = obj.get("Properties") or {}
                nodes[oid] = {
                    "name": props.get("name", oid),
                    "type": default_type,
                    "highvalue": bool(props.get("highvalue", False)),
                    "enabled": props.get("enabled", True),
                    "admincount": bool(props.get("admincount", False)),
                    "domain": props.get("domain", ""),
                }

                # --- ACL abuse: who controls THIS object? ---
                for ace in obj.get("Aces") or []:
                    right = ace.get("RightName")
                    principal = ace.get("PrincipalSID")
                    if right in relevant_rights and principal:
                        add_edge(principal, oid, right, inherited=ace.get("IsInherited", False))

                # --- Groups: Members -> MemberOf ---
                for member in obj.get("Members") or []:
                    member_sid = member.get("ObjectIdentifier")
                    if member_sid:
                        add_edge(member_sid, oid, "MemberOf")

                # --- Computers: local administrators -> AdminTo ---
                for admin in _safe_get(obj, "LocalAdmins", "Results", default=[]) or []:
                    admin_sid = admin.get("ObjectIdentifier")
                    if admin_sid:
                        add_edge(admin_sid, oid, "AdminTo")

                # --- Computers: active sessions -> HasSession (Computer -> User) ---
                for sess in _safe_get(obj, "Sessions", "Results", default=[]) or []:
                    comp_sid, user_sid = sess.get("ComputerSID"), sess.get("UserSID")
                    if comp_sid and user_sid:
                        add_edge(comp_sid, user_sid, "HasSession")

                # --- Kerberos constrained delegation -> AllowedToDelegate ---
                for target in obj.get("AllowedToDelegate") or []:
                    target_sid = target if isinstance(target, str) else target.get("ObjectIdentifier")
                    if target_sid:
                        add_edge(oid, target_sid, "AllowedToDelegate")

                stats["objects_parsed"] += 1

            except Exception as e:
                stats["parse_errors"] += 1
                warnings.warn(f"Skipped one object in {member_name} due to an error: {e}")
                continue

    if verbose:
        print(f"[parsing] {stats['objects_parsed']} objects processed, {stats['parse_errors']} errors skipped.")
        print(f"[parsing] Relationships extracted: {dict(stats['edges_by_type'])}")

    return nodes, edges, stats

### Loading the SharpHound export (strict, no fallback)

This production version does **not** generate or fall back to any sample data. If
`CONFIG["SHARPHOUND_ZIP_PATH"]` does not point to an existing file, the cell below raises a
`FileNotFoundError` with a clear message, instead of continuing with placeholder data.


In [ ]:
if not os.path.exists(CONFIG["SHARPHOUND_ZIP_PATH"]):
    raise FileNotFoundError(
        f"SharpHound export not found at '{CONFIG['SHARPHOUND_ZIP_PATH']}'. "
        f"Update CONFIG['SHARPHOUND_ZIP_PATH'] to point to your actual SharpHound .zip export "
        f"(or upload it to this path first, e.g. via the Colab file browser)."
    )

with open(CONFIG["SHARPHOUND_ZIP_PATH"], "rb") as f:
    SHARPHOUND_ZIP_BYTES = f.read()

print(f"Loaded SharpHound archive: {CONFIG['SHARPHOUND_ZIP_PATH']} ({len(SHARPHOUND_ZIP_BYTES) / 1024:.1f} KB)")

### Running — Section 2

In [ ]:
raw_streams, member_names = load_sharphound_zip(SHARPHOUND_ZIP_BYTES)
print("Files recognized in the archive:", member_names)

nodes, edges, parse_stats = extract_objects_and_edges(raw_streams, member_names)
print(f"\n{len(nodes)} AD objects extracted, {len(edges)} relationships extracted.")

## 3. Graph modeling (NetworkX)

We build a **directed multigraph** (`MultiDiGraph`): each node is an AD object (user, group,
machine), each edge a directed relationship `source --relation--> target` (e.g.
`alice --GenericAll--> svc_backup` means *"alice has full control over svc_backup"*).
A `MultiDiGraph` is required because two objects can be linked by **several distinct
relationships** at once (e.g. both `AdminTo` **and** `MemberOf` between the same two nodes).

**Graph cleanup** (noise reduction):
- removing **self-loops**, which are irrelevant for an attack path;
- removing **isolated nodes** (no incoming/outgoing relationship after filtering) — an AD object
  with no exploitable ACL and no group membership adds nothing to path analysis.

**High-value target detection**: we rely primarily on the `Properties.highvalue` flag that
BloodHound/SharpHound sets automatically (Domain Admins, Enterprise Admins, domain controllers,
`AdminSDHolder`-protected accounts...). Failing that (older collector versions without this
flag), we fall back to well-known RIDs (`-512` Domain Admins, `-519` Enterprise Admins, `-516`
Domain Controllers, `-498`, `-544` Administrators).

> ⚠️ **SharpHound never marks a node as "owned" (compromised)** — that's a feature of
> BloodHound's own GUI, absent from the raw JSON files. The analysis's starting point
> (`COMPROMISED_PRINCIPAL_HINT`) must therefore always be provided manually.


In [ ]:
def build_graph(nodes: dict, edges: list) -> nx.MultiDiGraph:
    '''Builds the directed graph from the nodes/edges extracted in section 2.'''
    G = nx.MultiDiGraph()
    for sid, attrs in nodes.items():
        G.add_node(sid, **attrs)

    for src, dst, relation, meta in edges:
        # A principal referenced by an ACE/Session but never seen as a full object (outside the
        # collection scope, e.g. a principal from another domain) -> "ghost" node
        for sid in (src, dst):
            if sid not in G:
                G.add_node(sid, name=sid, type="Unknown", highvalue=False, enabled=True,
                           admincount=False, domain="")
        G.add_edge(src, dst, key=relation, relation=relation, **meta)

    return G


def clean_graph(G: nx.MultiDiGraph, remove_isolated=True, remove_self_loops=True) -> nx.MultiDiGraph:
    '''Reduces noise: strips self-loops and fully isolated nodes.'''
    G = G.copy()
    if remove_self_loops:
        G.remove_edges_from(list(nx.selfloop_edges(G)))
    if remove_isolated:
        G.remove_nodes_from(list(nx.isolates(G)))
    return G


WELL_KNOWN_HIGH_VALUE_RID_SUFFIXES = ("-512", "-519", "-516", "-498", "-544")


def get_high_value_targets(G: nx.MultiDiGraph) -> set:
    '''High-value targets: BloodHound's 'highvalue' flag, or a well-known RID as a fallback.'''
    targets = set()
    for n, attrs in G.nodes(data=True):
        if attrs.get("highvalue") or n.endswith(WELL_KNOWN_HIGH_VALUE_RID_SUFFIXES):
            targets.add(n)
    return targets


def find_node_by_name(G: nx.MultiDiGraph, substring: str) -> list:
    '''Looks up nodes by substring of their name (case-insensitive) — handy since SIDs are not
    memorable.'''
    substring = substring.lower()
    return [n for n, a in G.nodes(data=True) if substring in a.get("name", "").lower()]


def resolve_source_node(G: nx.MultiDiGraph, hint: str) -> str:
    '''Resolves a user-provided hint (exact SID or substring of the name) into a single SID.'''
    if hint in G:
        return hint
    candidates = find_node_by_name(G, hint)
    if not candidates:
        raise ValueError(f"No node matches '{hint}'. Try another name or a full SID.")
    if len(candidates) > 1:
        names = [G.nodes[c]["name"] for c in candidates]
        print(f"[notice] Multiple matches for '{hint}': {names} -> using the 1st one.")
    return candidates[0]


def find_attack_paths(G: nx.MultiDiGraph, source: str, targets: set = None,
                       cutoff: int = 8, max_paths_per_target: int = 3) -> list:
    '''
    For each high-value target, looks for the shortest path(s) from `source`.
    Raises nx.NodeNotFound if `source` does not exist in the graph; returns [] (no exception)
    if no path exists to a given target.
    '''
    if source not in G:
        raise nx.NodeNotFound(f"Source node not found in the graph: {source}")
    if targets is None:
        targets = get_high_value_targets(G)
    targets = {t for t in targets if t in G and t != source}

    results = []
    for target in targets:
        try:
            paths = list(nx.all_shortest_paths(G, source, target))
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue
        for p in paths[:max_paths_per_target]:
            if cutoff and len(p) - 1 > cutoff:
                continue
            results.append(p)

    results.sort(key=len)
    return results

### Running — Section 3

In [ ]:
G_raw = build_graph(nodes, edges)
G = clean_graph(G_raw)
print(f"Raw graph    : {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")
print(f"Cleaned graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

high_value_targets = get_high_value_targets(G)
print(f"\n{len(high_value_targets)} high-value target(s) detected:")
for t in high_value_targets:
    print("  -", G.nodes[t]["name"])

SOURCE_NODE = resolve_source_node(G, CONFIG["COMPROMISED_PRINCIPAL_HINT"])
print(f"\nStarting point (compromised): {G.nodes[SOURCE_NODE]['name']}  [{SOURCE_NODE}]")

attack_paths = find_attack_paths(G, SOURCE_NODE, high_value_targets, cutoff=CONFIG["MAX_PATH_LENGTH"])
print(f"\n{len(attack_paths)} attack path(s) found:\n")
for p in attack_paths:
    print("  " + "  ->  ".join(G.nodes[n]["name"] for n in p))

## 4. Natural-language translation

An LLM understands a **descriptive sentence** far better than a list of `(SID, relation, SID)`
tuples. This section converts each graph path (`Node A -> GenericAll -> Node B -> AdminTo ->
Node C`) into a clear English paragraph, translating each relationship type into an explicit
phrase (the `RELATION_PHRASES_EN` dictionary, easily extensible).

We also prepare **token-budget management**: a large AD environment can surface dozens of attack
paths. To never exceed the LLM's context window when sending data in section 5,
`chunk_descriptions` groups descriptions into batches whose estimated size stays under a
configurable threshold.


In [ ]:
RELATION_PHRASES_EN = {
    "GenericAll": "has full control (GenericAll) over",
    "WriteDacl": "can modify the access permissions (WriteDacl) of",
    "ForceChangePassword": "can force a password reset (ForceChangePassword) of",
    "AdminTo": "has local administrator rights on",
    "HasSession": ("has an active session of the following user open on its machine, "
                   "potentially exposing their credentials:"),
    "AllowedToDelegate": "is allowed to delegate (Kerberos constrained delegation, AllowedToDelegate) to",
    "MemberOf": "is a member of the group",
    # Additional relationships recognized if RELEVANT_ACE_RIGHTS is extended (section 2):
    "Owns": "owns (Owns)",
    "GenericWrite": "can modify sensitive attributes (GenericWrite) of",
    "WriteOwner": "can change the owner (WriteOwner) of",
    "AddMember": "can add itself (AddMember) to the group",
    "AllowedToAct": "can authenticate on behalf of others via resource-based constrained "
                    "delegation (RBCD, AllowedToAct) on",
}


def _node_label(G, n):
    a = G.nodes[n]
    return f"{a.get('name', n)} ({a.get('type', '?')})"


def path_to_natural_language(G: nx.MultiDiGraph, path: list) -> str:
    '''Converts a path (list of SIDs) into an English paragraph describing each hop.'''
    if len(path) < 2:
        return f"The node {_node_label(G, path[0])} is not a path (only one node)."

    lines = [f"Starting point: {_node_label(G, path[0])}, considered compromised ('owned')."]
    for i in range(len(path) - 1):
        src, dst = path[i], path[i + 1]
        edge_data = G.get_edge_data(src, dst)
        relation = next(iter(edge_data.values()))["relation"] if edge_data else "UnknownRelation"
        phrase = RELATION_PHRASES_EN.get(relation, f"is linked by a '{relation}' relationship to")
        lines.append(f"- {_node_label(G, src)} {phrase} {_node_label(G, dst)}.")

    if G.nodes[path[-1]].get("highvalue"):
        lines.append(
            f"The final target, {_node_label(G, path[-1])}, is a high-value object "
            f"(sensitive account/group or domain controller)."
        )
    return "\n".join(lines)


def estimate_tokens(text: str) -> int:
    '''Portable heuristic (~4 characters/token, no tiktoken-style dependency).'''
    return max(1, len(text) // 4)


def chunk_descriptions(descriptions: list, max_tokens_per_batch: int = 3000,
                        reserved_for_prompt: int = 500) -> list:
    '''Groups descriptions into batches whose estimated size stays under the given budget,
    so the context window is never exceeded when sending a batch to the LLM.'''
    budget = max(max_tokens_per_batch - reserved_for_prompt, estimate_tokens(max(descriptions, key=len, default="")))
    batches, current, current_tokens = [], [], 0
    for desc in descriptions:
        t = estimate_tokens(desc)
        if current and current_tokens + t > budget:
            batches.append(current)
            current, current_tokens = [], 0
        current.append(desc)
        current_tokens += t
    if current:
        batches.append(current)
    return batches

### Running — Section 4

In [ ]:
path_descriptions = [path_to_natural_language(G, p) for p in attack_paths]

if path_descriptions:
    print("Example natural-language translation (first path found):\n")
    print(path_descriptions[0])
else:
    print("No attack path to translate (see section 3).")

## 5. LLM analysis (Red & Blue Team)

Each natural-language description is sent to an LLM with a **strict system prompt** demanding a
structured response. We aim for a dual reading:

- 🔴 **Red Team**: attack mechanics, MITRE ATT&CK technique, theoretical offensive tools;
- 🔵 **Blue Team / SOC**: concrete remediations and detection leads to break this specific path.

**Supported providers** (your choice, via `CONFIG["LLM_PROVIDER"]`):
- `groq` — **Groq** API (free tier, rate-limited), OpenAI-compatible endpoint, extremely low
  latency. Uses the **`openai`** SDK pointed at a different `base_url`. Key from
  [console.groq.com](https://console.groq.com/keys), environment variable `GROQ_API_KEY` (on
  Colab: secret `GROQ_API_KEY` in the 🔑 tab). An explicit `"groq"` choice automatically falls
  back to `mock` on failure (missing key, temporary outage);
- `anthropic` — Claude API (`pip install anthropic`, environment variable `ANTHROPIC_API_KEY`);
- `openai` — OpenAI API (`pip install openai`, environment variable `OPENAI_API_KEY`);
- `mock` — **no network call**, deterministic simulated responses. Useful for developing/testing
  the pipeline without burning API credits, or for running this notebook with no key at all.
- `"auto"` (not the default here) detects the 1st available key in the environment (Anthropic,
  then OpenAI, then Groq) and falls back to `mock` if none is found.

An alternative **LangChain** integration is also provided (optional, see below) for those who
prefer to standardize their LLM calls through that ecosystem.


In [ ]:
SYSTEM_PROMPT_EN = """You are a senior cybersecurity analyst specialized in Active Directory \
security (both Red Team AND Blue Team). You are given the natural-language description of ONE \
attack path extracted from a BloodHound/SharpHound collection (an ordered list of relationships \
between AD objects).

For EACH path provided, respond STRICTLY with a valid JSON object (no text outside the JSON, no \
```markdown``` fences) matching exactly this schema:

{
  "primary_technique": "Short name of the dominant technique (e.g. Kerberoasting, DCSync, Pass-the-Hash, ACL abuse, RBCD, Constrained Delegation)",
  "mitre_attack": "Relevant MITRE ATT&CK identifier(s), e.g. T1558.003",
  "attack_mechanics": "Concise technical explanation (3-5 sentences) of how the attack unfolds step by step along this path",
  "offensive_tools": ["List of public tools commonly used for this technique, e.g. Rubeus, Mimikatz, Impacket, BloodHound"],
  "remediation": ["Concrete remediation actions to break this specific path"],
  "soc_detection": ["Rules / Windows events / indicators to monitor to detect this path (e.g. Event ID, EDR/SIEM queries)"],
  "risk_level": "Critical | High | Medium | Low"
}

Constraints:
- Stay factual and technical, no marketing emphasis.
- Do not invent any relationship absent from the provided description.
- If some information is uncertain, state so explicitly in the relevant field rather than omitting it.
- The JSON must be directly parsable by json.loads() in Python.
"""

SYSTEM_PROMPT_BATCH_EN = SYSTEM_PROMPT_EN + """

IMPORTANT (batch mode): you are given several numbered paths in a single message, delimited by \
"### PATH n ###". You must respond with a JSON ARRAY containing EXACTLY one object per path \
received, IN THE SAME ORDER, each object following the schema described above. Never merge \
several paths into a single object. Do not return any text outside the JSON array.
"""

print("System prompts ready (single-path mode + batch mode).")

In [ ]:
class BaseLLMProvider:
    """Common interface for every LLM provider used in this notebook."""
    name = "base"

    def generate(self, system_prompt: str, user_prompt: str) -> str:
        raise NotImplementedError


class MockLLMProvider(BaseLLMProvider):
    """
    Dummy provider, with NO network call and NO API key: lets the whole pipeline run and be
    validated (demo, tests) even without access to a paid LLM API.
    Never use this for a real security analysis — the responses are static.
    """
    name = "mock"

    def _one(self, idx=1):
        return {
            "primary_technique": f"ACL abuse + session extraction (DEMO #{idx} — simulated response)",
            "mitre_attack": "T1098 / T1550",
            "attack_mechanics": (
                "[DEMO MODE - no real LLM called] The attacker exploits excessive ACL rights "
                "to take control of an intermediate account, obtains local administrator rights, "
                "then extracts the credentials of a privileged account present in a session on "
                "the compromised machine."
            ),
            "offensive_tools": ["BloodHound", "Mimikatz", "PowerView / PowerSploit"],
            "remediation": [
                "Apply least privilege to the ACLs of service accounts",
                "Forbid interactive sessions of privileged accounts on standard workstations (Tiering model)",
            ],
            "soc_detection": [
                "Monitor ACL/ACE changes (Event ID 5136)",
                "Detect privileged-account logons outside domain controllers / admin workstations",
            ],
            "risk_level": "Critical",
        }

    def generate(self, system_prompt: str, user_prompt: str) -> str:
        n_paths = len(re.findall(r"###\s*PATH\s+\d+\s*###", user_prompt))
        if n_paths >= 2:
            return json.dumps([self._one(i + 1) for i in range(n_paths)], ensure_ascii=False, indent=2)
        return json.dumps(self._one(), ensure_ascii=False, indent=2)


class AnthropicProvider(BaseLLMProvider):
    """Calls the Anthropic API (Claude). Requires `pip install anthropic`."""
    name = "anthropic"

    def __init__(self, api_key=None, model="claude-sonnet-5-5", max_tokens=1500):
        import anthropic  # deferred import: optional dependency
        self.client = anthropic.Anthropic(api_key=api_key)  # api_key=None -> reads ANTHROPIC_API_KEY
        self.model = model
        self.max_tokens = max_tokens

    def generate(self, system_prompt, user_prompt):
        resp = self.client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return "".join(block.text for block in resp.content if getattr(block, "type", None) == "text")


class OpenAIProvider(BaseLLMProvider):
    """Calls the OpenAI API (Chat Completions). Requires `pip install openai`."""
    name = "openai"

    def __init__(self, api_key=None, model="gpt-4o-mini", max_tokens=1500):
        import openai  # deferred import: optional dependency
        self.client = openai.OpenAI(api_key=api_key)  # api_key=None -> reads OPENAI_API_KEY
        self.model = model
        self.max_tokens = max_tokens

    def generate(self, system_prompt, user_prompt):
        resp = self.client.chat.completions.create(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return resp.choices[0].message.content


class GroqProvider(BaseLLMProvider):
    """
    Calls a model hosted on Groq (free tier, rate-limited) via the `openai` SDK: Groq's
    endpoint is compatible with OpenAI's Chat Completions API, so only the client's `base_url`
    changes. Requires `pip install openai`. Free API key at https://console.groq.com/keys
    """
    name = "groq"

    def __init__(self, api_key=None, model="llama-3.3-70b-versatile", max_tokens=1500):
        import openai  # deferred import: optional dependency (shared with OpenAIProvider)
        self.client = openai.OpenAI(
            base_url="https://api.groq.com/openai/v1",  # only the endpoint differs from OpenAI
            api_key=api_key,
        )
        self.model = model
        self.max_tokens = max_tokens

    def generate(self, system_prompt, user_prompt):
        resp = self.client.chat.completions.create(
            model=self.model,
            max_tokens=self.max_tokens,
            temperature=0.2,  # low temperature: favors strict, reproducible JSON output
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return resp.choices[0].message.content


def parse_llm_json(raw_text: str):
    """Cleans then parses the LLM response as JSON (tolerates leftover ```json ... ``` fences)."""
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, f"Invalid JSON returned by the LLM: {e}"


print("LLM providers defined: mock, anthropic, openai, groq.")

### 5.1 Alternative integration via LangChain (optional)

If you'd rather go through **LangChain** instead of the native SDKs (e.g. to standardize several
providers, add callbacks, caching, etc.), here is an equivalent wrapper. This cell is
**optional**: if `langchain` is not installed, the notebook keeps working normally with
`AnthropicProvider` / `OpenAIProvider` / `GroqProvider` / `MockLLMProvider` above.

```bash
pip install langchain langchain-anthropic langchain-openai langchain-groq
```


In [ ]:
class LangChainProvider(BaseLLMProvider):
    """
    Optional wrapper based on LangChain. Instantiate with backend="anthropic", "openai" or "groq".
    Only usable if the matching LangChain package is installed
    (`langchain-anthropic`, `langchain-openai`, or `langchain-groq`).
    """
    name = "langchain"

    def __init__(self, backend="anthropic", model=None, api_key=None, max_tokens=1500):
        if backend == "anthropic":
            from langchain_anthropic import ChatAnthropic
            self.llm = ChatAnthropic(model=model or "claude-sonnet-5-5", api_key=api_key, max_tokens=max_tokens)
        elif backend == "openai":
            from langchain_openai import ChatOpenAI
            self.llm = ChatOpenAI(model=model or "gpt-4o-mini", api_key=api_key, max_tokens=max_tokens)
        elif backend == "groq":
            from langchain_groq import ChatGroq
            self.llm = ChatGroq(model=model or "llama-3.3-70b-versatile", api_key=api_key, max_tokens=max_tokens)
        else:
            raise ValueError(f"Unknown LangChain backend: {backend}")

    def generate(self, system_prompt, user_prompt):
        from langchain_core.messages import SystemMessage, HumanMessage
        response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
        return response.content


try:
    import langchain_core  # noqa: F401
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False

print("LangChain available:", LANGCHAIN_AVAILABLE, "(optional — not required for the rest of the notebook)")

### 5.2 Orchestration: single, batched analysis, and context-budget management

Two complementary functions:

- `analyze_attack_path_with_llm`: analyzes **a single** path, with automatic retries
  (`retries`) on transient network/API errors.
- `analyze_paths_in_batches`: the main entry point. It splits the list of descriptions into
  batches via `chunk_descriptions` (section 4) to **never exceed the LLM's context window**, and:
  - a batch of 1 path → a regular call, response = a JSON object;
  - a batch of several paths → **a single LLM call** for the whole batch (saves tokens and
    latency), expected response = a JSON array of the same length;
  - if the returned array is invalid or the wrong size → **automatic fallback** to analyzing
    each path of the batch individually (slower, but reliable).


In [ ]:
def analyze_attack_path_with_llm(provider: BaseLLMProvider, path_description: str,
                                  system_prompt: str = SYSTEM_PROMPT_EN, retries: int = 1) -> dict:
    """Analyzes a single attack path. Always returns a dict (an 'error' key on failure)."""
    raw, last_err = None, None
    for attempt in range(retries + 1):
        try:
            raw = provider.generate(system_prompt, path_description)
            break
        except Exception as e:  # various network/API errors depending on the SDK used
            last_err = e
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))

    if raw is None:
        return {"error": f"LLM call failed ({provider.name}) after {retries + 1} attempt(s): {last_err}",
                "analyzed_path": path_description}

    parsed, err = parse_llm_json(raw)
    if err:
        return {"error": err, "raw_response": raw, "analyzed_path": path_description}
    parsed["analyzed_path"] = path_description
    return parsed


def _format_batch_prompt(descriptions: list) -> str:
    return "\n\n".join(f"### PATH {i + 1} ###\n{desc}" for i, desc in enumerate(descriptions))


def analyze_paths_in_batches(provider: BaseLLMProvider, descriptions: list,
                              max_tokens_per_batch: int = 3000, delay_between_calls: float = 0.0,
                              retries: int = 1) -> list:
    """
    Analyzes a list of paths, grouping them into batches under the given token budget.
    Always returns a list of the same length as `descriptions`, in the same order.
    """
    if not descriptions:
        return []

    batches = chunk_descriptions(descriptions, max_tokens_per_batch=max_tokens_per_batch)
    results = []

    for batch in batches:
        if len(batch) == 1:
            results.append(analyze_attack_path_with_llm(provider, batch[0], retries=retries))
            if delay_between_calls:
                time.sleep(delay_between_calls)
            continue

        batch_prompt = _format_batch_prompt(batch)
        raw, last_err = None, None
        for attempt in range(retries + 1):
            try:
                raw = provider.generate(SYSTEM_PROMPT_BATCH_EN, batch_prompt)
                break
            except Exception as e:
                last_err = e
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))

        if raw is None:
            warnings.warn(f"LLM call failed for a batch of {len(batch)} paths ({last_err}) -> falling back to individual calls.")
            results.extend(analyze_attack_path_with_llm(provider, d, retries=retries) for d in batch)
            continue

        parsed, err = parse_llm_json(raw)
        if err or not isinstance(parsed, list) or len(parsed) != len(batch):
            warnings.warn(
                f"Invalid or inconsistently sized batch response ({err or 'size differs from the batch'}) "
                f"-> falling back to individually analyzing the {len(batch)} paths in this batch."
            )
            results.extend(analyze_attack_path_with_llm(provider, d, retries=retries) for d in batch)
            continue

        for desc, item in zip(batch, parsed):
            item["analyzed_path"] = desc
            results.append(item)

        if delay_between_calls:
            time.sleep(delay_between_calls)

    return results


print("Orchestration functions ready: analyze_attack_path_with_llm, analyze_paths_in_batches.")

### 5.3 Automatic provider selection

In [ ]:
def select_llm_provider(config=CONFIG) -> BaseLLMProvider:
    """Chooses the LLM provider according to CONFIG['LLM_PROVIDER'], falling back to 'mock' if needed."""
    choice = config["LLM_PROVIDER"]

    def try_anthropic():
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("ANTHROPIC_API_KEY is missing from the environment.")
        return AnthropicProvider(api_key=ANTHROPIC_API_KEY, model=config["ANTHROPIC_MODEL"])

    def try_openai():
        if not OPENAI_API_KEY:
            raise RuntimeError("OPENAI_API_KEY is missing from the environment.")
        return OpenAIProvider(api_key=OPENAI_API_KEY, model=config["OPENAI_MODEL"])

    def try_groq():
        if not GROQ_API_KEY:
            raise RuntimeError("GROQ_API_KEY is missing from the environment (Colab secret or env var).")
        return GroqProvider(api_key=GROQ_API_KEY, model=config["GROQ_MODEL"])

    if choice == "anthropic":
        return try_anthropic()
    if choice == "openai":
        return try_openai()
    if choice == "groq":
        # Unlike the providers above, an explicit "groq" choice does NOT crash the notebook on
        # failure (missing key, SDK not installed...): Groq's free tier can be temporarily
        # unavailable (rate limits) -> fall back to Mock. Errors happening later, during the
        # real calls (section 5.2), are already handled by the automatic retries and the
        # "error" field in the results.
        try:
            return try_groq()
        except Exception as e:
            warnings.warn(f"Failed to initialize GroqProvider ({e}) -> falling back to MockLLMProvider.")
            return MockLLMProvider()
    if choice == "mock":
        return MockLLMProvider()

    # choice == "auto": try Anthropic, then OpenAI, then Groq, then fall back to mock
    for factory in (try_anthropic, try_openai, try_groq):
        try:
            return factory()
        except Exception:
            continue

    warnings.warn(
        "No valid LLM API key found in the environment "
        "(ANTHROPIC_API_KEY / OPENAI_API_KEY / GROQ_API_KEY). "
        "Falling back to MockLLMProvider: the analyses below are SIMULATED, not a real security analysis."
    )
    return MockLLMProvider()


llm_provider = select_llm_provider()
print(f"Selected LLM provider: {llm_provider.name}")

### Running — Section 5: full analysis of the detected paths

In [ ]:
if not path_descriptions:
    print("No path to analyze.")
    results_df = pd.DataFrame()
else:
    analysis_results = analyze_paths_in_batches(
        llm_provider,
        path_descriptions,
        max_tokens_per_batch=CONFIG["MAX_TOKENS_PER_LLM_CALL"],
    )

    n_errors = sum(1 for r in analysis_results if "error" in r)
    print(f"{len(analysis_results)} path(s) analyzed, {n_errors} error(s).")

    results_df = pd.DataFrame(analysis_results)
    display_cols = [c for c in ["primary_technique", "risk_level", "mitre_attack", "error"]
                     if c in results_df.columns]
    display(results_df[display_cols])

In [ ]:
# Full detail of the 1st result (attack mechanics, tools, remediations, detection)
if path_descriptions and analysis_results:
    first = analysis_results[0]
    print("=" * 100)
    print("ANALYZED PATH:")
    print(first.get("analyzed_path", "N/A"))
    print("-" * 100)
    if "error" in first:
        print("ERROR:", first["error"])
    else:
        print(f"Primary technique : {first.get('primary_technique')}")
        print(f"MITRE ATT&CK      : {first.get('mitre_attack')}")
        print(f"Risk level        : {first.get('risk_level')}")
        print(f"\nAttack mechanics:\n{first.get('attack_mechanics')}")
        print(f"\nTheoretical offensive tools:")
        for tool in first.get("offensive_tools", []):
            print(f"  - {tool}")
        print(f"\nRemediation (Blue Team):")
        for r in first.get("remediation", []):
            print(f"  - {r}")
        print(f"\nSOC detection:")
        for d in first.get("soc_detection", []):
            print(f"  - {d}")
    print("=" * 100)

## 6. Exporting results and visualization (bonus)

We export the analyses to CSV/JSON (a reusable deliverable for the capstone report or a SOC
ticket export), and draw the **subgraph** corresponding to the paths found (never the full graph
of a real AD: unreadable beyond a few dozen nodes).


In [ ]:
import matplotlib.pyplot as plt

os.makedirs("outputs", exist_ok=True)

if not results_df.empty:
    results_df.to_json("outputs/attack_path_analysis.json", orient="records", force_ascii=False, indent=2)
    results_df.to_csv("outputs/attack_path_analysis.csv", index=False)
    print("Results exported -> outputs/attack_path_analysis.{json,csv}")

if attack_paths:
    # Subgraph = union of all nodes/edges involved in the paths found
    path_nodes = {n for p in attack_paths for n in p}
    H = G.subgraph(path_nodes)

    color_map = {"User": "#4C9AFF", "Group": "#FFAB00", "Computer": "#36B37E", "Unknown": "#999999"}
    node_colors = [color_map.get(H.nodes[n].get("type"), "#999999") for n in H.nodes]
    labels = {n: H.nodes[n].get("name", n).split("@")[0] for n in H.nodes}

    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(H, seed=42, k=1.2)
    nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=1800, alpha=0.9)
    nx.draw_networkx_labels(H, pos, labels=labels, font_size=8)
    edge_labels = {(u, v): d.get("relation", "") for u, v, d in H.edges(data=True)}
    nx.draw_networkx_edges(H, pos, arrowstyle="-|>", arrowsize=15, connectionstyle="arc3,rad=0.1")
    nx.draw_networkx_edge_labels(H, pos, edge_labels=edge_labels, font_size=7, label_pos=0.5)
    plt.title("Subgraph of detected attack paths")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig("outputs/attack_path_subgraph.png", dpi=150)
    plt.show()
    print("Figure exported -> outputs/attack_path_subgraph.png")
else:
    print("No path to visualize.")

## Conclusion and known limitations

**Limitations worth documenting in your report (and possible improvements):**

- **Objects not modeled**: `PrimaryGroupSID` (primary group membership, requires recombining a
  RID with the domain SID), `OUs`/`GPOs`/`Containers` (attack paths via GPO inheritance),
  `AllowedToAct` (RBCD) — this last one is already supported by the translation dictionary
  (section 4) but not extracted by default in section 2; just add `"AllowedToAct"` to
  `RELEVANT_ACE_RIGHTS`, or write a dedicated extraction block (same pattern as
  `LocalAdmins`/`Sessions`).
- **High-value target detection**: depends on the collector's `highvalue` flag (absent from very
  old SharpHound versions) and a well-known-RID fallback list — always double-check the
  relevance of detected targets on a real AD.
- **"Owned" is not native to SharpHound**: the analysis's starting point must be provided
  manually (`CONFIG["COMPROMISED_PRINCIPAL_HINT"]`).
- **LLM reliability**: the outputs (tools, techniques, remediations) must be **reviewed by a
  human** before any action — an LLM can be wrong or hallucinate a technical detail, hence the
  value of strict JSON parsing and the `error`/`raw_response` fields on failure.
- **Rate limiting**: error handling here is deliberately simple (1 retry with light backoff). For
  heavy use, plan a real exponential-backoff strategy and respect your LLM provider's rate limits
  — this matters especially for Groq's free tier.
- **No built-in sample data**: this production version strictly requires a real SharpHound
  export; there is no fallback dataset to fall back on for a quick demo run.

> 🔁 **To reuse this notebook on a real AD**: place your SharpHound export at the path defined by
> `CONFIG["SHARPHOUND_ZIP_PATH"]`, adjust `CONFIG["COMPROMISED_PRINCIPAL_HINT"]` with your
> scenario's compromised account, export an API key (`export GROQ_API_KEY=...`, or set the Colab
> secret), then re-run the whole notebook (*Restart & Run All*).
